# Chinese Rap Lyrics NER Pipeline

端到端流水线：数据清洗 → NER 实体识别 → Bag-of-Entities → K-Means 聚类

每一步的中间结果都会缓存到 `outputs/` 目录。如果缓存存在则直接加载，跳过计算。
想要重新计算某一步，删掉对应的 CSV 文件再 run 那个 cell 即可。

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

OUTPUTS = Path("outputs")
OUTPUTS.mkdir(exist_ok=True)

## 1. 数据加载与清洗

缓存文件: `outputs/artist_lyrics.csv`

清洗步骤：
- 移除 Live 版本、伴奏/instrumental 等非原创录音室版本
- 从歌词文本中剥离制作人信息行（出品、Prod.、混音、母带等）
- 剥离说话人标签（小老虎：、周士爵：等）
- 移除版权声明文本
- 同一艺人文本去重

In [ ]:
ARTIST_LYRICS_PATH = OUTPUTS / "artist_lyrics.csv"

if ARTIST_LYRICS_PATH.exists():
    print(f"[CACHE] Loading from {ARTIST_LYRICS_PATH}")
    artist_lyrics = pd.read_csv(ARTIST_LYRICS_PATH)
    print(f"Loaded: {len(artist_lyrics)} artists")
else:
    from src.data_cleaning import load_and_clean, combine_by_artist

    df_raw = load_and_clean("lyrics_chunks_enriched.csv")
    artist_lyrics = combine_by_artist(df_raw)
    artist_lyrics.to_csv(ARTIST_LYRICS_PATH, index=False)
    print(f"\n[SAVED] {ARTIST_LYRICS_PATH}")

In [ ]:
# 浏览数据
print(f"共 {len(artist_lyrics)} 位艺人")

artist_lyrics["text_len"] = artist_lyrics["combined_text"].str.len()
print(f"\n歌词长度分布:")
print(artist_lyrics["text_len"].describe())

top10 = artist_lyrics.nlargest(10, "text_len")[["artist", "text_len"]]
print(f"\n歌词最长的 10 位艺人:")
for _, r in top10.iterrows():
    print(f"  {r['artist']}: {r['text_len']:,} 字符")

## 2. NER 实体识别

缓存文件: `outputs/entities_long.csv`

**模型选择**：
- `zh_core_web_trf`（默认）— transformer 模型，精度最高，需要 `torch` + `spacy-transformers`
- `zh_core_web_lg` — CNN 模型，速度快，不需要 torch

安装：
```bash
pip install spacy-transformers torch
python -m spacy download zh_core_web_trf
# 或者
python -m spacy download zh_core_web_lg
```

这一步最耗时（全量 241 位艺人）。跑完后结果自动缓存，之后直接加载。
想换模型时，删掉 `outputs/entities_long.csv` 再 run。

In [ ]:
from src.ner import build_nlp, extract_entities, normalize_entities

# 切换模型：改这一行即可
SPACY_MODEL = "zh_core_web_trf"  # or "zh_core_web_lg"

ENTITIES_PATH = OUTPUTS / "entities_long.csv"
ENTITIES_RAW_PATH = OUTPUTS / "entities_raw.csv"  # pre-normalization backup

if ENTITIES_PATH.exists():
    print(f"[CACHE] Loading from {ENTITIES_PATH}")
    entity_df = pd.read_csv(ENTITIES_PATH)
    print(f"Loaded: {len(entity_df)} entity mentions")
else:
    import gc

    nlp = build_nlp("configs/rap_lexicon_seed.jsonl", model_name=SPACY_MODEL)
    entity_df = extract_entities(artist_lyrics, nlp, min_entity_len=2)

    # Free the spaCy model from memory now that NER is done
    del nlp
    gc.collect()
    print("[INFO] Released spaCy model from memory")

    # Save raw entities before normalization
    entity_df.to_csv(ENTITIES_RAW_PATH, index=False)

    # Normalize: merge space artifacts, case variants, suffix forms, substrings
    entity_df = normalize_entities(entity_df)
    entity_df.to_csv(ENTITIES_PATH, index=False)
    print(f"\n[SAVED] {ENTITIES_PATH}")

In [ ]:
from src.ner import entity_summary

# Top 30 全局高频实体
print("=== Top 30 高频实体 ===")
top30 = entity_summary(entity_df, top_n=30)
print(top30.to_string(index=False))

In [ ]:
# 每个标签类型的实体数量
print("=== 标签分布 ===")
label_dist = entity_df["label"].value_counts()
print(label_dist)

# 每种标签的 top-5 实体
print("\n=== 各标签类型 Top-5 实体 ===")
for label in label_dist.index:
    subset = entity_df[entity_df["label"] == label]
    top5 = subset["entity"].value_counts().head(5)
    print(f"\n[{label}] ({len(subset)} mentions)")
    for ent, cnt in top5.items():
        print(f"  {ent}: {cnt}")

## 2.5 实体人工审核与修正

审核文件: `configs/entity_review.csv`

**首次运行**：自动生成审核 CSV（每个标签 top 100 实体）。
**审核方式**：打开 CSV，填写 `action` 列：
- 留空 → 保留原样
- `delete` → 删除该实体
- 填写标签名（如 `CITY`）→ 改为该标签

**全局兜底**：`GLOBAL_MIN_COUNT` 参数确保全局高频实体不会因为在本类别排不进
top-k 而被遗漏。例如设为 10，则任何出现 ≥10 次的实体都会被纳入审核范围。

**审核完成后**：重新 Run 下面的 cell，修正会自动应用。
已审核的 `action` 不会丢失——重新生成时只追加新增实体。
需要完全重新审核时，删掉 `configs/entity_review.csv` 再 run。

In [ ]:
from src.ner import generate_entity_review, apply_entity_corrections

REVIEW_PATH = "configs/entity_review.csv"
MIN_COUNT = 2  # auto-delete entities not in review with count < 2

# Generate review CSV only if it doesn't exist yet
if not Path(REVIEW_PATH).exists():
    GLOBAL_MIN_COUNT = 10  # also review any entity with global freq >= this
    generate_entity_review(entity_df, output_path=REVIEW_PATH, top_n=100,
                           global_min_count=GLOBAL_MIN_COUNT)
    print("\n>>> 请打开 configs/entity_review.csv 填写 action 列，然后重新 Run 这个 cell <<<")
else:
    # Apply corrections + auto-delete low-frequency unreviewed entities
    n_before = len(entity_df)
    entity_df = apply_entity_corrections(entity_df, review_path=REVIEW_PATH, min_count=MIN_COUNT)
    if len(entity_df) != n_before:
        # Overwrite cached entities with corrected version
        entity_df.to_csv(ENTITIES_PATH, index=False)
        print(f"[SAVED] Updated {ENTITIES_PATH}")
        # Clear clustering cache so it uses corrected entities
        for f in OUTPUTS.glob("bag_of_entities.*"):
            f.unlink()
        for f in [OUTPUTS / "artist_clusters.csv", OUTPUTS / "cluster_entity_summary.csv"]:
            if f.exists():
                f.unlink()
        print("[INFO] Cleared clustering cache (will recompute with corrected entities)")

## 3. Bag-of-Entities 矩阵与聚类

缓存文件: `outputs/bag_of_entities.npz` (稀疏格式), `outputs/artist_clusters.csv`, `outputs/cluster_entity_summary.csv`

矩阵使用 scipy 稀疏格式存储，内存占用减少约 200 倍（99%+ 的元素是零）。

In [ ]:
from src.clustering import EntityMatrix, build_bag_of_entities, run_kmeans, summarize_clusters

BOE_PATH = OUTPUTS / "bag_of_entities"  # .npz + .artists.csv + .entities.csv
CLUSTERS_PATH = OUTPUTS / "artist_clusters.csv"
SUMMARY_PATH = OUTPUTS / "cluster_entity_summary.csv"

N_CLUSTERS = 6

if (OUTPUTS / "bag_of_entities.npz").exists() and CLUSTERS_PATH.exists() and SUMMARY_PATH.exists():
    print("[CACHE] Loading clustering results (sparse format)")
    entity_matrix = EntityMatrix.load_npz(str(BOE_PATH))
    assignments = pd.read_csv(CLUSTERS_PATH)
    summaries = pd.read_csv(SUMMARY_PATH)
    print(f"Loaded: {entity_matrix.shape[0]} artists x {entity_matrix.shape[1]} entities, "
          f"{assignments['cluster'].nunique()} clusters")
    print(f"Sparse matrix: {entity_matrix.data.nnz} non-zero entries "
          f"({entity_matrix.data.nnz / (entity_matrix.shape[0] * entity_matrix.shape[1]) * 100:.1f}% dense)")
else:
    import gc

    entity_matrix = build_bag_of_entities(entity_df)
    nnz = entity_matrix.data.nnz
    total = entity_matrix.shape[0] * entity_matrix.shape[1]
    print(f"Bag-of-Entities: {entity_matrix.shape[0]} artists x {entity_matrix.shape[1]} entities")
    print(f"Non-zero: {nnz} / {total} ({nnz/total*100:.1f}% dense, {(1 - nnz/total)*100:.1f}% sparse)")

    assignments, centers = run_kmeans(entity_matrix, n_clusters=N_CLUSTERS, random_state=42)
    summaries = summarize_clusters(centers, entity_matrix.entities, top_k=15)
    del centers
    gc.collect()

    entity_matrix.save_npz(str(BOE_PATH))
    assignments.to_csv(CLUSTERS_PATH, index=False)
    summaries.to_csv(SUMMARY_PATH, index=False)
    print(f"\n[SAVED] bag_of_entities.npz, {CLUSTERS_PATH.name}, {SUMMARY_PATH.name}")

In [ ]:
# 各聚类的艺人
print(f"=== 聚类结果（K={assignments['cluster'].nunique()}）===")
for cluster_id in sorted(assignments["cluster"].unique()):
    artists_in_cluster = assignments[assignments["cluster"] == cluster_id]["artist"].tolist()
    print(f"\n--- Cluster {cluster_id} ({len(artists_in_cluster)} artists) ---")
    print(", ".join(artists_in_cluster))

In [ ]:
# 各聚类代表实体
print("=== 各聚类代表实体（Top 15）===")
for cluster_name in summaries["cluster"].unique():
    cluster_data = summaries[summaries["cluster"] == cluster_name]
    cluster_data = cluster_data[cluster_data["centroid_weight"] > 0]
    print(f"\n--- {cluster_name} ---")
    for _, r in cluster_data.iterrows():
        print(f"  {r['entity']}: {r['centroid_weight']:.4f}")

## 4. 质量检查

抽样检查实体识别的效果，帮助发现 false positive / false negative。

In [ ]:
# 单个艺人的实体详细检查
CHECK_ARTIST = artist_lyrics.iloc[0]["artist"]
print(f"=== 质量检查: {CHECK_ARTIST} ===")

artist_entities = entity_df[entity_df["artist"] == CHECK_ARTIST]
print(f"该艺人共提取 {len(artist_entities)} 个实体 mention")
print(f"\n实体频率:")
for (ent, label), cnt in artist_entities.groupby(["entity", "label"]).size().sort_values(ascending=False).head(20).items():
    print(f"  {ent} ({label}): {cnt}")

# 显示该艺人歌词片段以便人工核验
text_sample = artist_lyrics[artist_lyrics["artist"] == CHECK_ARTIST]["combined_text"].iloc[0]
print(f"\n歌词片段（前 500 字）:")
print(text_sample[:500])

In [ ]:
# 可疑实体检查
print("=== 可疑实体（可能是噪声）===")

entity_counts = entity_df["entity"].value_counts()
singletons = entity_counts[entity_counts == 1]
print(f"\n只出现 1 次的实体数: {len(singletons)} / {len(entity_counts)} ({len(singletons)/len(entity_counts)*100:.1f}%)")
print("样本:", singletons.head(20).index.tolist())

long_entities = entity_df[entity_df["entity"].str.len() > 10]["entity"].unique()
print(f"\n长度 > 10 的实体 ({len(long_entities)} 个):")
for e in long_entities[:20]:
    print(f"  \"{e}\"")